In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "catalog_project")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storage-account", "adlssmartdata1921")

In [0]:
container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage-account")

file_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/cliente.json"


In [0]:

clientschema = StructType(fields=[
    StructField("idProducto", IntegerType(), True),
    StructField("idCliente", IntegerType(), True),
    StructField("Nombre", StringType(), True),
    StructField("excenta", BooleanType(), True),
])

In [0]:

dfclient = spark.read \
    .schema(clientschema) \
    .option("multiLine", True) \
    .json(file_path)

In [0]:
dfclient.display()

In [0]:
dfclientdate = dfclient.withColumn("ingestion_date", current_timestamp())

In [0]:
client_df = dfclientdate.select(
    "idProducto",
    "idCliente",
    "Nombre",
    "excenta",
    "ingestion_date"
)


In [0]:
client_df.display()

In [0]:
client_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.client")